# Homework — Yearly Salary vs Experience

Dataset: [Salary Dataset — Simple Linear Regression](https://www.kaggle.com/datasets/abhishek14398/salary-dataset-simple-linear-regression)

Goal: predict `Salary` from `YearsExperience` with a simple linear regression, and report how accurate the model is.

Task checklist: **A)** clean data · **B)** build model · **C)** measure accuracy · **D)** document (this notebook).

## A. Load & Clean the Data

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pd.set_option("display.precision", 2)

In [2]:
raw = pd.read_csv("../data/salary_dataset.csv")
raw.head()

,Unnamed: 0,YearsExperience,Salary
0,0,1.2,39344.0
1,1,1.4,46206.0
2,2,1.6,37732.0
3,3,2.1,43526.0
4,4,2.3,39892.0


In [3]:
# The CSV ships an unnamed row-index column we don't need.
df = raw.drop(columns=[c for c in raw.columns if c.startswith("Unnamed")])

print("shape:", df.shape)
print("missing values:\n", df.isna().sum())
print("duplicate rows:", df.duplicated().sum())
df.describe()
df.head(10)

shape: (30, 2)
missing values:
 YearsExperience    0
Salary             0
dtype: int64
duplicate rows: 0


,YearsExperience,Salary
0,1.2,39344.0
1,1.4,46206.0
2,1.6,37732.0
3,2.1,43526.0
4,2.3,39892.0
5,3.0,56643.0
6,3.1,60151.0
7,3.3,54446.0
8,3.3,64446.0
9,3.8,57190.0


No missing values and no duplicate rows — the dataset is already clean, so cleaning here is just the confirmation above plus dropping the stray index column.

## Quick Look at the Data

In [4]:
scatter = go.Figure()
scatter.add_trace(go.Scatter(
    x=df["YearsExperience"], y=df["Salary"], mode="markers",
    marker=dict(size=10, color="#2563eb"), name="employees",
    hovertemplate="Experience: %{x} yrs<br>Salary: $%{y:,.0f}<extra></extra>",
))
scatter.update_layout(title="Salary vs Years of Experience", xaxis_title="Years of experience",
                       yaxis_title="Salary ($)", template="plotly_white", width=750, height=480)
scatter.show()

Salary rises roughly in a straight line with experience — a good candidate for linear regression.

## B. Build the Linear Regression Model

In [5]:
X = df[["YearsExperience"]]
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"slope (coef_):    {model.coef_[0]:.2f}")
print(f"intercept:        {model.intercept_:.2f}")

slope (coef_):    9423.82
intercept:        24380.20


In [6]:
line_x = np.linspace(X["YearsExperience"].min() - 0.5, X["YearsExperience"].max() + 0.5, 50)
line_y = model.predict(line_x.reshape(-1, 1))

fit_figure = go.Figure()
fit_figure.add_trace(go.Scatter(x=X_train["YearsExperience"], y=y_train, mode="markers",
                                 marker=dict(size=10, color="#2563eb"), name="train"))
fit_figure.add_trace(go.Scatter(x=X_test["YearsExperience"], y=y_test, mode="markers",
                                 marker=dict(size=10, color="#16a34a", symbol="diamond"), name="test"))
fit_figure.add_trace(go.Scatter(x=line_x, y=line_y, mode="lines",
                                 line=dict(color="#dc2626", width=3), name="fitted line"))
fit_figure.update_layout(title="Fitted Regression Line", xaxis_title="Years of experience",
                          yaxis_title="Salary ($)", template="plotly_white", width=750, height=480)
fit_figure.show()

/home/akbar/akbarDev/hbai/academy-tutorials/linear-regression/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


## C. Measure the Model's Accuracy

In [9]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"R^2:  {r2:.3f}   (fraction of variance in Salary explained by YearsExperience)")
print(f"MAE:  ${mae:,.0f}   (average absolute prediction error)")
print(f"RMSE: ${rmse:,.0f}   (penalizes big misses more)")

R^2:  0.902   (fraction of variance in Salary explained by YearsExperience)
MAE:  $6,286   (average absolute prediction error)
RMSE: $7,059   (penalizes big misses more)


In [8]:
residual_figure = go.Figure()
residual_figure.add_trace(go.Scatter(x=y_test, y=y_pred, mode="markers",
                                      marker=dict(size=10, color="#2563eb"), name="test predictions"))
diag = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
residual_figure.add_trace(go.Scatter(x=diag, y=diag, mode="lines",
                                      line=dict(color="#dc2626", dash="dash"), name="perfect prediction"))
residual_figure.update_layout(title="Predicted vs Actual Salary (test set)", xaxis_title="Actual salary ($)",
                               yaxis_title="Predicted salary ($)", template="plotly_white", width=650, height=480)
residual_figure.show()

## D. Summary

- **Data**: 30 rows, one feature (`YearsExperience`) → one target (`Salary`). No missing values or duplicates;
  only the stray unnamed index column was dropped.
- **Model**: ordinary least-squares linear regression (`sklearn.linear_model.LinearRegression`),
  trained on an 80/20 train/test split.
- **Accuracy**: R², MAE and RMSE are printed above. R² close to 1 and the points hugging the diagonal in the
  predicted-vs-actual plot both mean experience is an almost perfect linear predictor of salary in this dataset.
- **Takeaway**: with a small, clean, near-linear dataset like this one, a single-feature linear model already
  fits very well — the "hard part" of this exercise was setup, not model complexity.